In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import requests
import ast
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths

# SharePoint
path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'Census Data')
path_main = os.path.join(path_sp, 'Data')

# Git
if user == 'jfontes':
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')
if user in ['jchoy', 'AAlAzzawi']:
    path_git     = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_config0 = os.path.join(path_git, 'config')
    path_code    = os.path.join(path_git, 'Python Code', 'Census')
    path_config  = os.path.join(path_code, 'aa_config')

In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

## Set API key
# Obtain API Key from the following source 
# https://api.census.gov/data/key_signup.html
# Copy retrieved API key to .txt file for safe keeping
file_api = open(os.path.join(path_config, 'api_key.txt'))
api_key = file_api.read()
file_api.close()

In [ ]:
# Use URL to county fips mapping table
# Import county FIPS codes by state
url_fips = "https://www2.census.gov/geo/docs/reference/codes/files/national_county.txt"
df_fips = pd.read_csv(url_fips, header = None, sep = ',')
df_fips.head()

In [ ]:
# Clean and subset FIPS file
# rename columns
# clean county name
# reformat FIPS field

df_fips.columns = ['State', 'State FIPS', 'County FIPS', 'County Name', 'to be removed']
df_fips = df_fips[['State', 'State FIPS']].drop_duplicates()

df_fips = df_fips[df_fips['State FIPS'] < 60]
df_fips = df_fips.reset_index(drop = True)
df_fips['State FIPS' ] = df_fips['State FIPS' ].astype(str).apply('{:0>2}'.format)

# show
df_fips.head()

In [ ]:
df_fips

In [ ]:
states = df_fips['State FIPS'].unique()
states = ['53', '54', '55', '56']
states

In [ ]:
start_time = time.time()

year_start = 2009
year_end   = 2022
years_to_import = range(year_start, year_end+1)

g_ = '?get='

# User inputs for user API key, desired variables and years to import
api_key_ = f"&key={api_key}"
variables_ = 'NAME'

# list_df_census = []


print('Importing place IDs for all states...')
print('')

for state in states:
    print(state)
    for year in tqdm(years_to_import):
        root_ = f'https://api.census.gov/data/{year}/acs/acs5'
    
        # Specify which geography to import
        location_ = '&for=place:*' + '&in=state:' + state
        
        ## Concatenate constructed URL
        query = f"{root_}{g_}{variables_}{location_}{api_key_}"

        ## Call data using URL
    
        # Use requests package to call out to the API
        response = requests.get(query).text
        response = ast.literal_eval(response)
        
        # convert parsed response text to pandas df
        df_census = pd.DataFrame(response[1:], columns = response[0])
        
        # apply year tag
        df_census['Year'] = year

        list_df_census.append(df_census)


print('')
print('Concatenating all states together...')

df_places = pd.concat(list_df_census)


print("")
print("Finished!!")
print(f"Process complete.  It took --- {round((time.time() - start_time)/60, 1)} minutes ---")

In [ ]:
df_places

In [ ]:
df_test1 = df_places[['NAME', 'state', 'place']].drop_duplicates()
df_test2 = df_places[df_places['Year'] == 2022]

In [ ]:
df_test1

In [ ]:
df_test2

In [ ]:
df_test1[df_test1.NAME.duplicated()].head(20)

In [ ]:
df_places[df_places['NAME'] == 'Lily Lake CDP, Wisconsin']
df_places[df_places['NAME'] == 'Fox Park CDP, Wyoming'   ]
df_places[df_places['NAME'] == 'Watkins CDP, Colorado'   ]
df_places[df_places['NAME'] == 'Temescal Valley CDP, California']


# it looks like they change PLACE IDS over time but mostly stay consistent
# just use the latest year to save space on the configurration file

In [ ]:
df_places_out = df_places.copy()


df_places_out = df_places_out[df_places_out['Year'] == 2022]


# Remove anything after specified string, use regular expression (currently set to remove everything after the first period)
def re_remove_post(x, exp = ','):
    if x == 'nan':
        return 'nan'
    else:
        return x.split(exp, 1)[0]
        
df_places_out['NAME'] = df_places_out['NAME'].apply(re_remove_post)




In [ ]:
df_places_out.to_excel(os.path.join(path_config0, 'States to Places ID Mapping.xlsx'), index = False)